# 03 · Preprocessing, step by step

This notebook walks through every Phase 2 cleaning step on **one day file** so each step's effect is visible. It calls the same functions as the scripts (`src/data/labels.py`, `src/data/preprocess.py`, `src/features/feature_pipeline.py`). The notebook and the pipeline therefore can't drift apart: if a step changes, it changes in one place.

The full run over all 8 files is `python -m src.data.preprocess` followed by `python -m src.features.feature_pipeline`. The last cell shows its results.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
pd.set_option('display.max_columns', 20)

## 1. Load one file with clean column names

The headers have leading spaces (`' Flow Duration'`), and `Fwd Header Length` appears twice. `normalize_columns` strips the spaces and renames the second copy to `Fwd Header Length.1`, so no column is lost silently.

In [2]:
from src.data.io import discover_files, scan_file, read_raw_header, normalize_columns, find_label_column
source = next(s for s in discover_files('data/raw') if 'WebAttacks' in s.name)
scan = scan_file(source)
raw_header = read_raw_header(source, scan.encoding)
norm = normalize_columns(raw_header)
print(source.name, scan.encoding)
print('renamed duplicates:', norm.deduplicated)
with source.open_text(scan.encoding) as fh:
    df = pd.read_csv(fh, header=0, names=norm.names, low_memory=False)
label_col = find_label_column(norm.names)
features = [c for c in df.columns if c != label_col]
df.shape

Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv utf-8
renamed duplicates: [('Fwd Header Length', 'Fwd Header Length.1')]


(170366, 79)

## 2. Normalise labels

The web-attack labels contain `�`, because the original dash was lost before the dataset was published. `normalize_labels` restores it, adds the attack **family**, and adds the binary target **is_attack**. An unknown label raises an error instead of being guessed.

In [3]:
from src.data.labels import normalize_labels
meta = normalize_labels(df[label_col])
pd.DataFrame({'raw': df[label_col], 'label': meta['label'], 'family': meta['family'], 'is_attack': meta['is_attack']}).value_counts().to_frame('rows')

,,,,rows
raw,label,family,is_attack,
BENIGN,BENIGN,BENIGN,0,168186
Web Attack � Brute Force,Web Attack - Brute Force,Web Attack,1,1507
Web Attack � XSS,Web Attack - XSS,Web Attack,1,652
Web Attack � Sql Injection,Web Attack - Sql Injection,Web Attack,1,21


## 3. Exact duplicates

If identical rows end up in both train and test, the test score partly measures memorisation. We drop rows whose features **and** label are identical, keeping the first copy. Rows with the same features but *different* labels are kept, because that ambiguity is real.

In [4]:
dup = df.duplicated()
print(f'{dup.sum():,} exact duplicate rows out of {len(df):,}')
meta.loc[dup, 'label'].value_counts().to_frame('duplicates removed')

6,066 exact duplicate rows out of 170,366


,duplicates removed
label,
BENIGN,6029
Web Attack - Brute Force,37


In [5]:
same_features = df.duplicated(subset=features, keep=False)
conflicts = df[same_features].groupby(features, dropna=False)[label_col].nunique()
print('feature vectors with more than one label in this file:', int((conflicts > 1).sum()))
df = df[~dup].reset_index(drop=True); meta = meta[~dup].reset_index(drop=True)
df.shape

feature vectors with more than one label in this file: 0


(164300, 79)

## 4. Infinity and missing values

These appear only in two rate columns, where a flow's duration is 0 and the rate is computed as bytes / 0. The pipeline turns ±inf into NaN and fills it with the **training** median. Clipping to an arbitrary maximum would invent a value.

In [6]:
X = df[features]
pd.DataFrame({'nan': X.isna().sum(), '+inf': np.isposinf(X).sum(), '-inf': np.isneginf(X).sum()}).query('nan + `+inf` + `-inf` > 0')

,nan,+inf,-inf
Flow Bytes/s,19,102,0
Flow Packets/s,0,121,0


## 5. Split before fitting anything

Every learned step comes after the split. That includes which columns are constant, which columns are duplicates, the medians, and the scaling. In the real run the split is stratified 70/15/15 over all 8 files; here it's one file, to keep the demo small.

In [7]:
from src.data.preprocess import assign_split
split = assign_split(meta['label'], seed=42)
pd.crosstab(meta['label'], split)

col_0,test,train,val
label,,,
BENIGN,24324,113510,24323
Web Attack - Brute Force,220,1029,221
Web Attack - Sql Injection,3,15,3
Web Attack - XSS,98,456,98


## 6. The fitted pipeline

`build_preprocessor()` is a scikit-learn `Pipeline`:

1. `SchemaGuard`: fixes the feature names and order, and rejects missing or non-numeric columns
2. `ExcludeColumns`: drops `Destination Port`, because in this dataset the port almost reveals the label
3. ±inf → NaN
4. `DropUninformative`: drops columns that are constant or exact copies **in the training data**
5. median imputation
6. (scaled variant only) signed log + `StandardScaler`, for Logistic Regression

It is fitted on train only, then applied unchanged to validation and test.

In [8]:
from src.features.feature_pipeline import build_preprocessor, describe
train_X = X[split == 'train']
pipe = build_preprocessor().fit(train_X)
info = describe(pipe)
print('excluded:', info['excluded'])
print('dropped (constant):', info['dropped_constant'])
print('dropped (duplicate of):', info['dropped_duplicate_of'])
print(f"{len(info['input_features'])} input features -> {len(info['output_features'])} model features")

excluded: ['Destination Port']
dropped (constant): ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
dropped (duplicate of): {'SYN Flag Count': 'Fwd PSH Flags', 'ECE Flag Count': 'RST Flag Count', 'Avg Fwd Segment Size': 'Fwd Packet Length Mean', 'Fwd Header Length.1': 'Fwd Header Length', 'Subflow Fwd Packets': 'Total Fwd Packets', 'Subflow Fwd Bytes': 'Total Length of Fwd Packets', 'Subflow Bwd Packets': 'Total Backward Packets'}
78 input features -> 60 model features


In [9]:
pipe.transform(X[split == 'test']).describe().T.head(10)

,count,mean,std,min,25%,50%,75%,max
Flow Duration,24645.0,1.287174e+07,3.246907e+07,-1.0,205.00,35197.0,1.230536e+06,1.199999e+08
Total Fwd Packets,24645.0,1.623628e+01,1.276964e+03,1.0,1.00,2.0,4.000000e+00,1.990570e+05
Total Backward Packets,24645.0,1.954474e+01,1.730057e+03,0.0,1.00,2.0,2.000000e+00,2.696190e+05
Total Length of Fwd Packets,24645.0,5.322410e+02,7.951917e+03,0.0,33.00,68.0,1.560000e+02,1.186087e+06
Total Length of Bwd Packets,24645.0,3.381779e+04,3.743834e+06,0.0,6.00,142.0,3.440000e+02,5.840000e+08
Fwd Packet Length Max,24645.0,1.673783e+02,4.070845e+02,0.0,28.00,41.0,6.400000e+01,1.022000e+04
Fwd Packet Length Min,24645.0,2.299619e+01,4.170069e+01,0.0,0.00,6.0,4.200000e+01,1.472000e+03
Fwd Packet Length Mean,24645.0,4.850385e+01,9.163896e+01,0.0,13.75,38.0,5.033333e+01,2.305263e+03
Fwd Packet Length Std,24645.0,4.730334e+01,1.336647e+02,0.0,0.00,0.0,2.012461e+01,2.772782e+03
Bwd Packet Length Max,24645.0,4.144523e+02,8.188028e+02,0.0,6.00,91.0,1.830000e+02,1.022000e+04


## 7. Results of the full run (all 8 files)

These are produced by `python -m src.data.preprocess` and `python -m src.features.feature_pipeline`.

In [10]:
import json
summary = json.load(open('reports/metrics/preprocessing/summary.json'))
print(f"rows in {summary['rows_in']:,} -> after de-duplication {summary['rows_out']:,}")
print('split sizes:', summary['split_rows'])
pd.read_csv('reports/metrics/preprocessing/label_by_split.csv', index_col=0)

rows in 2,830,743 -> after de-duplication 2,522,362
split sizes: {'train': 1765653, 'val': 378354, 'test': 378355}


,train,val,test
label,,,
BENIGN,1467539,314472,314473
Bot,1367,293,293
DDoS,89611,19202,19203
DoS GoldenEye,7200,1543,1543
DoS Hulk,120994,25927,25928
DoS Slowhttptest,3660,784,784
DoS slowloris,3770,807,808
FTP-Patator,4153,890,890
Heartbleed,8,2,1
